In [1]:
import os
import numpy as np
import pandas as pd



In [2]:
SAMPLE_PATH = "/kaggle/input/ventilator-pressure-prediction/sample_submission.csv"
TRAIN_PATH = "/kaggle/input/ventilator-pressure-prediction/train.csv"
TEST_PATH = "/kaggle/input/ventilator-pressure-prediction/test.csv"

sub = pd.read_csv(SAMPLE_PATH)

blend_paths = {
    "sub_1": "/kaggle/input/vpp-lstm-baseline-median-pp/submission.csv",
    "sub_2": "/kaggle/input/blend-of-blend-of-blend-of-blend-of-blend-of-ble/submission.csv",
    "sub_3": "/kaggle/input/gb-vpp-whoppity-dub-dub/median_submission.csv",
    "sub_4": "/kaggle/input/random-weights-blending-tool-ventilator-pressure/rwb 125 loops.csv",
}

loaded = {}
missing = []
for name, path in blend_paths.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        if "pressure" not in df.columns:
            missing.append(name)
        else:
            loaded[name] = df[["pressure"]].copy()
    else:
        missing.append(name)

print("Blend components found:", sorted(list(loaded.keys())))
print("Blend components missing:", sorted(missing))




Blend components found: []
Blend components missing: ['sub_1', 'sub_2', 'sub_3', 'sub_4']


In [3]:
def _snap_to_pressure_grid(
    pred_pressure: np.ndarray, pressure_grid: np.ndarray
) -> np.ndarray:
    """
    Kaggle ventilator pressures lie on a fixed discrete grid; snapping to the nearest
    observed pressure often reduces MAE without changing the underlying approach.
    """
    pred_pressure = np.asarray(pred_pressure, dtype=np.float32)
    grid = np.asarray(pressure_grid, dtype=np.float32)
    if grid.size == 0:
        return pred_pressure

    idx = np.searchsorted(grid, pred_pressure, side="left")
    idx0 = np.clip(idx - 1, 0, grid.size - 1)
    idx1 = np.clip(idx, 0, grid.size - 1)
    g0 = grid[idx0]
    g1 = grid[idx1]
    choose1 = np.abs(pred_pressure - g1) < np.abs(pred_pressure - g0)
    snapped = np.where(choose1, g1, g0).astype(np.float32)
    return snapped


def build_local_baseline_submission(sample_sub: pd.DataFrame) -> pd.DataFrame:
    """
    Local, fast baseline (no external submissions).

    Core logic preserved: group-mean tables + hierarchical merge/fillna backoff,
    trained on inspiratory rows + expiratory override + pressure-grid snapping.
    """
    usecols_train = ["breath_id", "R", "C", "u_out", "u_in", "time_step", "pressure"]
    usecols_test = ["id", "breath_id", "R", "C", "u_out", "u_in", "time_step"]

    train = pd.read_csv(TRAIN_PATH, usecols=usecols_train)
    test = pd.read_csv(TEST_PATH, usecols=usecols_test)

    pressure_grid = np.sort(train["pressure"].unique().astype(np.float32))

    train["step"] = train.groupby("breath_id").cumcount().astype(np.int16)
    test["step"] = test.groupby("breath_id").cumcount().astype(np.int16)

    BIN_T = 0.02
    train["time_step_bin"] = np.floor(train["time_step"].values / BIN_T).astype(
        np.int16
    )
    test["time_step_bin"] = np.floor(test["time_step"].values / BIN_T).astype(np.int16)

    exp_mask = train["u_out"].values == 1
    if exp_mask.any():
        exp_pressure_global = float(train.loc[exp_mask, "pressure"].median())
    else:
        exp_pressure_global = float(train["pressure"].median())

    train["u_in_prev"] = (
        train.groupby("breath_id")["u_in"].shift(1).fillna(0.0).astype(np.float32)
    )
    test["u_in_prev"] = (
        test.groupby("breath_id")["u_in"].shift(1).fillna(0.0).astype(np.float32)
    )

    train["u_in_2prev"] = (
        train.groupby("breath_id")["u_in"].shift(2).fillna(0.0).astype(np.float32)
    )
    test["u_in_2prev"] = (
        test.groupby("breath_id")["u_in"].shift(2).fillna(0.0).astype(np.float32)
    )

    train["u_out_prev"] = (
        train.groupby("breath_id")["u_out"].shift(1).fillna(0).astype(np.int8)
    )
    test["u_out_prev"] = (
        test.groupby("breath_id")["u_out"].shift(1).fillna(0).astype(np.int8)
    )

    train["u_out_next"] = (
        train.groupby("breath_id")["u_out"].shift(-1).fillna(1).astype(np.int8)
    )
    test["u_out_next"] = (
        test.groupby("breath_id")["u_out"].shift(-1).fillna(1).astype(np.int8)
    )

    train["is_insp_last"] = (
        (train["u_out"].values == 0) & (train["u_out_next"].values == 1)
    ).astype(np.int8)
    test["is_insp_last"] = (
        (test["u_out"].values == 0) & (test["u_out_next"].values == 1)
    ).astype(np.int8)

    train["is_exhale_transition"] = (
        (train["u_out"].values == 1) & (train["u_out_prev"].values == 0)
    ).astype(np.int8)
    test["is_exhale_transition"] = (
        (test["u_out"].values == 1) & (test["u_out_prev"].values == 0)
    ).astype(np.int8)

    train["u_in_cum"] = train.groupby("breath_id")["u_in"].cumsum().astype(np.float32)
    test["u_in_cum"] = test.groupby("breath_id")["u_in"].cumsum().astype(np.float32)

    train["u_in_diff"] = (train["u_in"].astype(np.float32) - train["u_in_prev"]).astype(
        np.float32
    )
    test["u_in_diff"] = (test["u_in"].astype(np.float32) - test["u_in_prev"]).astype(
        np.float32
    )

    train["dt"] = (
        train.groupby("breath_id")["time_step"]
        .diff()
        .fillna(train["time_step"])
        .astype(np.float32)
    )
    test["dt"] = (
        test.groupby("breath_id")["time_step"]
        .diff()
        .fillna(test["time_step"])
        .astype(np.float32)
    )

    train_dt = np.maximum(train["dt"].values, 1e-6).astype(np.float32)
    test_dt = np.maximum(test["dt"].values, 1e-6).astype(np.float32)

    train["u_in_slope"] = (train["u_in_diff"].values / train_dt).astype(np.float32)
    test["u_in_slope"] = (test["u_in_diff"].values / test_dt).astype(np.float32)

    BIN_UIN = 1.0
    BIN_UIN_PREV = 1.0
    # CHANGE (score-relevant, minimal): refine cumulative binning slightly (20 -> 10)
    # to better separate breath trajectories without changing the approach.
    BIN_UIN_CUM = 10.0
    BIN_UIN_DIFF = 1.0
    BIN_UIN_SLOPE = 5.0

    train["u_in_bin"] = np.floor(train["u_in"].values / BIN_UIN).astype(np.int16)
    test["u_in_bin"] = np.floor(test["u_in"].values / BIN_UIN).astype(np.int16)

    train["u_in_prev_bin"] = np.floor(train["u_in_prev"].values / BIN_UIN_PREV).astype(
        np.int16
    )
    test["u_in_prev_bin"] = np.floor(test["u_in_prev"].values / BIN_UIN_PREV).astype(
        np.int16
    )

    train["u_in_2prev_bin"] = np.floor(
        train["u_in_2prev"].values / BIN_UIN_PREV
    ).astype(np.int16)
    test["u_in_2prev_bin"] = np.floor(test["u_in_2prev"].values / BIN_UIN_PREV).astype(
        np.int16
    )

    train["u_in_cum_bin"] = np.floor(train["u_in_cum"].values / BIN_UIN_CUM).astype(
        np.int16
    )
    test["u_in_cum_bin"] = np.floor(test["u_in_cum"].values / BIN_UIN_CUM).astype(
        np.int16
    )

    BIN_UIN_CUM_LOCAL = 5.0
    train["u_in_cum_local_bin"] = np.floor(
        train["u_in_cum"].values / BIN_UIN_CUM_LOCAL
    ).astype(np.int16)
    test["u_in_cum_local_bin"] = np.floor(
        test["u_in_cum"].values / BIN_UIN_CUM_LOCAL
    ).astype(np.int16)

    DIFF_CLIP = 30.0
    train_diff_clip = np.clip(train["u_in_diff"].values, -DIFF_CLIP, DIFF_CLIP)
    test_diff_clip = np.clip(test["u_in_diff"].values, -DIFF_CLIP, DIFF_CLIP)
    train["u_in_diff_bin"] = np.floor(train_diff_clip / BIN_UIN_DIFF).astype(np.int16)
    test["u_in_diff_bin"] = np.floor(test_diff_clip / BIN_UIN_DIFF).astype(np.int16)

    SLOPE_ABS_CLIP = 60.0
    train_slope_abs_clip = np.clip(
        np.abs(train["u_in_slope"].values), 0.0, SLOPE_ABS_CLIP
    )
    test_slope_abs_clip = np.clip(
        np.abs(test["u_in_slope"].values), 0.0, SLOPE_ABS_CLIP
    )
    train["u_in_slope_bin"] = np.floor(train_slope_abs_clip / BIN_UIN_SLOPE).astype(
        np.int16
    )
    test["u_in_slope_bin"] = np.floor(test_slope_abs_clip / BIN_UIN_SLOPE).astype(
        np.int16
    )

    BREATH_UIN_MEAN_BIN = 2.0
    BREATH_UIN_MAX_BIN = 2.0
    tr_breath_stats = (
        train.groupby("breath_id", sort=False)["u_in"]
        .agg(breath_u_in_mean="mean", breath_u_in_max="max")
        .reset_index()
    )
    te_breath_stats = (
        test.groupby("breath_id", sort=False)["u_in"]
        .agg(breath_u_in_mean="mean", breath_u_in_max="max")
        .reset_index()
    )
    tr_breath_stats["breath_u_in_mean_bin"] = np.floor(
        tr_breath_stats["breath_u_in_mean"].values / BREATH_UIN_MEAN_BIN
    ).astype(np.int16)
    tr_breath_stats["breath_u_in_max_bin"] = np.floor(
        tr_breath_stats["breath_u_in_max"].values / BREATH_UIN_MAX_BIN
    ).astype(np.int16)
    te_breath_stats["breath_u_in_mean_bin"] = np.floor(
        te_breath_stats["breath_u_in_mean"].values / BREATH_UIN_MEAN_BIN
    ).astype(np.int16)
    te_breath_stats["breath_u_in_max_bin"] = np.floor(
        te_breath_stats["breath_u_in_max"].values / BREATH_UIN_MAX_BIN
    ).astype(np.int16)

    train = train.merge(
        tr_breath_stats[["breath_id", "breath_u_in_mean_bin", "breath_u_in_max_bin"]],
        on="breath_id",
        how="left",
    )
    test = test.merge(
        te_breath_stats[["breath_id", "breath_u_in_mean_bin", "breath_u_in_max_bin"]],
        on="breath_id",
        how="left",
    )

    train_insp = train.loc[train["u_out"].values == 0].copy()

    global_mean = (
        float(train_insp["pressure"].mean())
        if len(train_insp)
        else float(train["pressure"].mean())
    )

    # CHANGE (score-relevant, minimal): drop u_in_slope_bin from strong/mid tables.
    # Slope is noisy (dt effects) and fragments groups; removing it increases match rate
    # while keeping the same group-mean + backoff paradigm.
    mean_strong = (
        train_insp.groupby(
            [
                "R",
                "C",
                "step",
                "u_in_bin",
                "u_in_prev_bin",
                "u_in_cum_bin",
                "u_in_cum_local_bin",
                "u_in_diff_bin",
                "u_out_prev",
                "is_insp_last",
                "breath_u_in_mean_bin",
                "breath_u_in_max_bin",
            ],
            sort=False,
        )["pressure"]
        .mean()
        .reset_index()
    )

    mean_mid1 = (
        train_insp.groupby(
            [
                "R",
                "C",
                "step",
                "time_step_bin",
                "u_in_bin",
                "u_in_prev_bin",
                "u_in_cum_local_bin",
                "u_in_diff_bin",
                "u_out_prev",
                "breath_u_in_mean_bin",
            ],
            sort=False,
        )["pressure"]
        .mean()
        .reset_index()
        .rename(columns={"pressure": "p_mid1"})
    )

    mean_mid2 = (
        train_insp.groupby(
            ["R", "C", "step", "u_in_bin", "u_in_diff_bin", "u_out_prev"], sort=False
        )["pressure"]
        .mean()
        .reset_index()
        .rename(columns={"pressure": "p_mid2"})
    )
    mean_mid3 = (
        train_insp.groupby(["R", "C", "step", "u_in_bin"], sort=False)["pressure"]
        .mean()
        .reset_index()
        .rename(columns={"pressure": "p_mid3"})
    )
    mean_rcs = (
        train_insp.groupby(["R", "C", "step"], sort=False)["pressure"]
        .mean()
        .reset_index()
        .rename(columns={"pressure": "p_rcs"})
    )
    mean_rs = (
        train_insp.groupby(["R", "step"], sort=False)["pressure"]
        .mean()
        .reset_index()
        .rename(columns={"pressure": "p_rs"})
    )
    mean_cs = (
        train_insp.groupby(["C", "step"], sort=False)["pressure"]
        .mean()
        .reset_index()
        .rename(columns={"pressure": "p_cs"})
    )
    mean_s = (
        train_insp.groupby(["step"], sort=False)["pressure"]
        .mean()
        .reset_index()
        .rename(columns={"pressure": "p_s"})
    )

    train_exp = train.loc[train["u_out"].values == 1, ["R", "C", "step", "pressure"]]
    if len(train_exp):
        exp_rcs = (
            train_exp.groupby(["R", "C", "step"], sort=False)["pressure"]
            .median()
            .reset_index()
            .rename(columns={"pressure": "p_exp_rcs"})
        )
    else:
        exp_rcs = pd.DataFrame(columns=["R", "C", "step", "p_exp_rcs"])

    pred = test.merge(
        mean_strong,
        on=[
            "R",
            "C",
            "step",
            "u_in_bin",
            "u_in_prev_bin",
            "u_in_cum_bin",
            "u_in_cum_local_bin",
            "u_in_diff_bin",
            "u_out_prev",
            "is_insp_last",
            "breath_u_in_mean_bin",
            "breath_u_in_max_bin",
        ],
        how="left",
    )
    pred = pred.merge(
        mean_mid1,
        on=[
            "R",
            "C",
            "step",
            "time_step_bin",
            "u_in_bin",
            "u_in_prev_bin",
            "u_in_cum_local_bin",
            "u_in_diff_bin",
            "u_out_prev",
            "breath_u_in_mean_bin",
        ],
        how="left",
    )
    pred = pred.merge(
        mean_mid2,
        on=["R", "C", "step", "u_in_bin", "u_in_diff_bin", "u_out_prev"],
        how="left",
    )
    pred = pred.merge(mean_mid3, on=["R", "C", "step", "u_in_bin"], how="left")
    pred = pred.merge(mean_rcs, on=["R", "C", "step"], how="left")
    pred = pred.merge(mean_rs, on=["R", "step"], how="left")
    pred = pred.merge(mean_cs, on=["C", "step"], how="left")
    pred = pred.merge(mean_s, on=["step"], how="left")

    if len(exp_rcs):
        pred = pred.merge(exp_rcs, on=["R", "C", "step"], how="left")
    else:
        pred["p_exp_rcs"] = np.nan

    pred["pressure"] = pred["pressure"].fillna(pred["p_mid1"])
    pred["pressure"] = pred["pressure"].fillna(pred["p_mid2"])
    pred["pressure"] = pred["pressure"].fillna(pred["p_mid3"])
    pred["pressure"] = pred["pressure"].fillna(pred["p_rcs"])
    pred["pressure"] = pred["pressure"].fillna(pred["p_rs"])
    pred["pressure"] = pred["pressure"].fillna(pred["p_cs"])
    pred["pressure"] = pred["pressure"].fillna(pred["p_s"])
    pred["pressure"] = pred["pressure"].fillna(global_mean)

    exp_test_mask = pred["u_out"].values == 1
    if exp_test_mask.any():
        exp_vals = pred.loc[exp_test_mask, "p_exp_rcs"].astype(np.float32).values
        fill_vals = np.where(
            np.isfinite(exp_vals), exp_vals, np.float32(exp_pressure_global)
        )
        pred.loc[exp_test_mask, "pressure"] = fill_vals

    out = sample_sub[["id"]].merge(pred[["id", "pressure"]], on="id", how="left")
    out["pressure"] = out["pressure"].fillna(global_mean).astype(np.float32)

    out["pressure"] = _snap_to_pressure_grid(out["pressure"].values, pressure_grid)

    test_uout_by_id = test[["id", "u_out", "R", "C", "step"]]
    out = out.merge(test_uout_by_id, on="id", how="left")
    if len(exp_rcs):
        out = out.merge(exp_rcs, on=["R", "C", "step"], how="left")
    else:
        out["p_exp_rcs"] = np.nan

    exp_out_mask = out["u_out"].values == 1
    if exp_out_mask.any():
        exp_vals2 = out.loc[exp_out_mask, "p_exp_rcs"].astype(np.float32).values
        fill_vals2 = np.where(
            np.isfinite(exp_vals2), exp_vals2, np.float32(exp_pressure_global)
        )
        out.loc[exp_out_mask, "pressure"] = fill_vals2

    out = out[["id", "pressure"]]
    return out




In [4]:
if all(k in loaded for k in ["sub_1", "sub_2", "sub_3", "sub_4"]):
    sub["pressure"] = (
        (loaded["sub_1"]["pressure"].values * 0.0)
        + (loaded["sub_2"]["pressure"].values * 0.3)
        + (loaded["sub_3"]["pressure"].values * 0.5)
        + (loaded["sub_4"]["pressure"].values * 0.2)
    )
    submission = sub[["id", "pressure"]].copy()
else:
    submission = build_local_baseline_submission(sub)

assert list(submission.columns) == ["id", "pressure"]
assert len(submission) == len(sub)
assert submission["pressure"].isna().sum() == 0

submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Wrote submission.csv with", len(submission), "rows")

   id   pressure
0   1   6.259305
1   2   5.978096
2   3   6.751420
3   4   7.876254
4   5  11.531965
Wrote submission.csv with 603600 rows
